In [1]:
import numpy as np
import pprint

class C45DecisionTree:
    def __init__(self, max_depth=5):
        self.max_depth = max_depth
        self.tree = None
        self.feature_names = None

    ### la moyenne
    def _entropy(self, y):
        if len(y) == 0: return 0
        proportions = np.bincount(y) / len(y)
        return -np.sum([p * np.log2(p) for p in proportions if p > 0])

    def _split_info(self, X_column):
        values, counts = np.unique(X_column, return_counts=True)
        proportions = counts / len(X_column)
        return -np.sum([p * np.log2(p) for p in proportions if p > 0])

    def _gain_ratio(self, X_column, y):
        parent_entropy = self._entropy(y)
        values, counts = np.unique(X_column, return_counts=True)
        
        weighted_entropy = 0
        for v, c in zip(values, counts):
            weighted_entropy += (c / len(y)) * self._entropy(y[X_column == v])
        
        info_gain = parent_entropy - weighted_entropy
        si = self._split_info(X_column)
        return info_gain / si if si != 0 else 0

    def fit(self, X, y, feature_names):
        self.feature_names = feature_names
        self.tree = self._build_tree(X, y, feature_names)

    def _build_tree(self, X, y, current_features, depth=0):
        # Cas de base : tous les labels sont identiques
        if len(np.unique(y)) == 1:
            return np.unique(y)[0]
        
        # Cas de base : plus de colonnes ou profondeur max
        if len(current_features) == 0 or depth >= self.max_depth:
            return np.bincount(y).argmax()

        # Sélection de la meilleure colonne via Gain Ratio
        gains = [self._gain_ratio(X[:, i], y) for i in range(X.shape[1])]
        best_idx = np.argmax(gains)
        best_name = current_features[best_idx]

        tree = {best_name: {}}
        values = np.unique(X[:, best_idx])

        for v in values:
            mask = X[:, best_idx] == v
            if len(y[mask]) == 0:
                tree[best_name][v] = np.bincount(y).argmax()
            else:
                # On prépare les données pour la branche suivante (on retire la colonne utilisée)
                new_X = np.delete(X[mask], best_idx, axis=1)
                new_features = [f for i, f in enumerate(current_features) if i != best_idx]
                tree[best_name][v] = self._build_tree(new_X, y[mask], new_features, depth + 1)
        return tree

    def predict_one(self, tree, sample, feature_names):
        if not isinstance(tree, dict):
            return tree
        
        root = list(tree.keys())[0]
        f_idx = feature_names.index(root)
        val = sample[f_idx]
        
        if val in tree[root]:
            return self.predict_one(tree[root][val], sample, feature_names)
        else:
            return 1 # Valeur par défaut si branche inconnue

# --- 1. PRÉPARATION DES DONNÉES ---
features_list = ["Outlook", "Temp", "Humidity", "Wind"]

# Encodage numérique :
# Outlook: Sunny=0, Overcast=1, Rain=2 | Temp: Hot=0, Mild=1, Cool=2
# Humidity: High=0, Normal=1 | Wind: Weak=0, Strong=1
X = np.array([
    [0,0,0,0], [0,0,0,1], [1,0,0,0], [2,1,0,0], [2,2,1,0],
    [2,2,1,1], [1,2,1,1], [0,1,0,0], [0,2,1,0], [2,1,1,0],
    [0,1,1,1], [1,1,0,1], [1,0,1,0], [2,1,0,1]
])
y = np.array([0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0]) # 0=No, 1=Yes

# --- 2. ENTRAÎNEMENT ---
c45 = C45DecisionTree(max_depth=5)
c45.fit(X, y, features_list)

print("Structure de l'arbre générée :")
pprint.pprint(c45.tree)

# --- 3. PHASE DE TEST ---
traduction = {
    "Outlook": {0: "Soleil", 1: "Nuageux", 2: "Pluie"},
    "Temp": {0: "Chaud", 1: "Bon", 2: "Frais"},
    "Hum": {0: "Haute", 1: "Normale"},
    "Wind": {0: "Faible", 1: "Fort"}
}

# Test sur un nouveau jour : Pluie (2), Bon (1), Normale (1), Faible (0)
nouveau_jour = [2, 1, 1, 0]

resultat = c45.predict_one(c45.tree, nouveau_jour, features_list)
verdict = "OUI, on joue !" if resultat == 1 else "NON, on ne joue pas."

print("\n" + "="*40)
print("RÉSULTAT DU TEST (C4.5)")
print("="*40)
print(f"Météo : {traduction['Outlook'][nouveau_jour[0]]}, Température {traduction['Temp'][nouveau_jour[1]]}")
print(f"Humidité {traduction['Hum'][nouveau_jour[2]]}, Vent {traduction['Wind'][nouveau_jour[3]]}")
print(f"Décision finale : {verdict}")
print("="*40)

Structure de l'arbre générée :
{'Outlook': {np.int64(0): {'Humidity': {np.int64(0): np.int64(0),
                                        np.int64(1): np.int64(1)}},
             np.int64(1): np.int64(1),
             np.int64(2): {'Wind': {np.int64(0): np.int64(1),
                                    np.int64(1): np.int64(0)}}}}

RÉSULTAT DU TEST (C4.5)
Météo : Pluie, Température Bon
Humidité Normale, Vent Faible
Décision finale : OUI, on joue !
